In [ ]:
Raw_Basis = "3T_healthy.basis"
Output_basis = "3T_basis_healthy_v1.h5"

target_bandwidth=939.85
target_n_timepoints=288

# Metabolite-specific removal of residual artifacts near 0 ppm.
#
# zero_below_ppm:
#     Spectrum is exactly zero below this value.
#
# keep_above_ppm:
#     Spectrum remains fully unchanged above this value.
#
# Between both limits, a smooth cosine transition is applied.

# LOW_PPM_ARTIFACT_CORRECTIONS = {
#     "AMLEA": {
#         "zero_below_ppm": 0.30,
#         "keep_above_ppm": 0.50,
#     },
#     "Ctn": {
#         "zero_below_ppm": 0.30,
#         "keep_above_ppm": 0.50,
#     },
#     "Try": {
#         "zero_below_ppm": 0.30,
#         "keep_above_ppm": 0.50,
#     },
# }

In [ ]:
from pathlib import Path
import sys


project_root = Path("..").resolve()

src_path = project_root / "src"

if not src_path.exists():
    raise FileNotFoundError(
        f"Could not find src directory: {src_path}"
    )

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from walinet.training_data.lcmodel_basis.parser import *

basis = load_lcmodel_basis(
    f"LCModelBasis/raw/{Raw_Basis}",
)

print(f"Metabolites : {basis.n_metabolites}")
print(f"Time points : {basis.n_points}")
print(f"Dwell time  : {basis.dwell_time:.9e} s")
print(f"Bandwidth   : {basis.bandwidth:.2f} Hz")
print(f"Hz / ppm    : {basis.hz_per_ppm:.3f}")

In [ ]:
# correction

from walinet.training_data.lcmodel_basis.hlsvd import (
    process_lcmodel_basis,
)


processed_basis = process_lcmodel_basis(
    basis,
    ppm_limits=(-0.2, 0.2),
    ppm_reference=4.65,
    n_singular_values=5,
    n_fit_points=8192,
)


# processed_basis = (
#     apply_low_ppm_artifact_corrections(
#         processed_basis=processed_basis,
#         basis=basis,
#         corrections=(
#             LOW_PPM_ARTIFACT_CORRECTIONS
#         ),
#     )
# )


print()
print("Finished.")

print(
    "Maximum reconstruction error:",
    (
        abs(
            processed_basis.original_fids
            - (
                processed_basis.clean_fids
                + processed_basis.reference_fids
            )
        )
    ).max(),
)

In [ ]:
print()
print("Finished.")

print(
    "Maximum reconstruction error:",
    (
        abs(
            processed_basis.original_fids
            - (
                processed_basis.clean_fids
                + processed_basis.reference_fids
            )
        )
    ).max(),
)

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_basis_before_after_grid,
)

plot_basis_before_after_grid(
    basis,
    processed_basis,
    ppm_limits=(7.5, -1),
)

In [ ]:
from pathlib import Path

from walinet.training_data.lcmodel_basis.library import (
    build_or_extend_basis_library,
)


basis_path = (
    project_root
    / "MetabModes/LCModelBasis"
    / f"raw/{Raw_Basis}"
)

output_library_path = (
    project_root
    / "MetabModes/LCModelBasis"
    / "processed"
    / f"{Output_basis}"
)


build_or_extend_basis_library(
    output_library_path,
    source_basis_path=basis_path,
    basis=basis,
    processed_basis=processed_basis,
    duplicate_policy="error",
    processing_repository_path=project_root,
)

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_basis_library_consistency,
)


plot_basis_library_consistency(
    f"/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/{Output_basis}",
    ppm_limits=(7.5, 0.0),
    n_columns=4,
)

In [ ]:
from walinet.training_data.lcmodel_basis.acquisition import (
    prepare_basis_for_acquisition,
)


prepared_basis = prepare_basis_for_acquisition(
    f"/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/{Output_basis}",
    target_bandwidth=target_bandwidth,
    target_n_timepoints=target_n_timepoints,
)

In [ ]:
prepared_basis.fids.shape

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_prepared_basis_grid,
)


plot_prepared_basis_grid(
    prepared_basis,
    ppm_limits=(7.5, 0),
    n_columns=4,
)

In [ ]:
# from walinet.training_data.lcmodel_basis.acquisition import (
#     prepare_basis_for_acquisition,
# )


# prepared_basis_tumor = prepare_basis_for_acquisition(
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/walinet_7T_native_basis_tumor_v1.h5",
#     target_bandwidth=2778.0,
#     target_n_timepoints=840,
# )

# prepared_basis_healthy = prepare_basis_for_acquisition(
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/walinet_7T_native_basis_v1.h5",
#     target_bandwidth=2778.0,
#     target_n_timepoints=840,
# )

In [ ]:
# import math

# import matplotlib.pyplot as plt
# import numpy as np


# def compare_prepared_bases(
#     basis_tumor,
#     basis_healthy,
#     component: str = "real",
#     n_columns: int = 4,
#     ppm_min: float | None = 0.0,
#     ppm_max: float | None = 5.0,
#     figsize_per_panel: tuple[float, float] = (4.0, 2.8),
# ):
#     """
#     Compare all metabolites occurring in both prepared LCModel bases.

#     Tumor basis:
#         solid line

#     Healthy basis:
#         dashed line

#     Parameters
#     ----------
#     basis_tumor:
#         Prepared tumor LCModel basis.

#     basis_healthy:
#         Prepared healthy LCModel basis.

#     component:
#         "real", "imag", or "abs".

#     n_columns:
#         Number of plot columns.

#     ppm_min, ppm_max:
#         Displayed ppm range. Set both to None for the full range.

#     figsize_per_panel:
#         Width and height per raster panel.
#     """
#     if component not in {"real", "imag", "abs"}:
#         raise ValueError(
#             "component must be 'real', 'imag', or 'abs'."
#         )

#     def normalize_name(name) -> str:
#         if isinstance(name, bytes):
#             name = name.decode("utf-8")

#         return str(name).strip()

#     tumor_names = [
#         normalize_name(name)
#         for name in basis_tumor.names
#     ]

#     healthy_names = [
#         normalize_name(name)
#         for name in basis_healthy.names
#     ]

#     tumor_lookup = {
#         name: index
#         for index, name in enumerate(tumor_names)
#     }

#     healthy_lookup = {
#         name: index
#         for index, name in enumerate(healthy_names)
#     }

#     common_metabolites = sorted(
#         set(tumor_names)
#         & set(healthy_names)
#     )

#     tumor_only = sorted(
#         set(tumor_names)
#         - set(healthy_names)
#     )

#     healthy_only = sorted(
#         set(healthy_names)
#         - set(tumor_names)
#     )

#     if not common_metabolites:
#         raise RuntimeError(
#             "No common metabolites found."
#         )

#     tumor_fids = np.asarray(
#         basis_tumor.fids
#     )

#     healthy_fids = np.asarray(
#         basis_healthy.fids
#     )

#     n_timepoints = tumor_fids.shape[-1]

#     if healthy_fids.shape[-1] != n_timepoints:
#         raise ValueError(
#             "Both bases must have the same number "
#             "of timepoints."
#         )

#     # Convert time-domain FIDs to fft-shifted spectra.
#     tumor_spectra = np.fft.fftshift(
#         np.fft.fft(
#             tumor_fids,
#             axis=-1,
#         ),
#         axes=-1,
#     )

#     healthy_spectra = np.fft.fftshift(
#         np.fft.fft(
#             healthy_fids,
#             axis=-1,
#         ),
#         axes=-1,
#     )

#     dwell_time = float(
#         basis_tumor.dwell_time
#     )

#     hz_per_ppm = float(
#         basis_tumor.hz_per_ppm
#     )

#     ppm_reference = float(
#         basis_tumor.ppm_reference
#     )

#     frequency_axis_hz = np.fft.fftshift(
#         np.fft.fftfreq(
#             n_timepoints,
#             d=dwell_time,
#         )
#     )

#     ppm_axis = (
#         ppm_reference
#         - frequency_axis_hz / hz_per_ppm
#     )

#     def select_component(spectrum):
#         if component == "real":
#             return spectrum.real

#         if component == "imag":
#             return spectrum.imag

#         return np.abs(spectrum)

#     n_metabolites = len(
#         common_metabolites
#     )

#     n_rows = math.ceil(
#         n_metabolites / n_columns
#     )

#     figure, axes = plt.subplots(
#         nrows=n_rows,
#         ncols=n_columns,
#         figsize=(
#             figsize_per_panel[0] * n_columns,
#             figsize_per_panel[1] * n_rows,
#         ),
#         squeeze=False,
#     )

#     flat_axes = axes.ravel()

#     amplitude_ratios = {}

#     for plot_index, metabolite in enumerate(
#         common_metabolites
#     ):
#         axis = flat_axes[plot_index]

#         tumor_spectrum = select_component(
#             tumor_spectra[
#                 tumor_lookup[metabolite]
#             ]
#         )

#         healthy_spectrum = select_component(
#             healthy_spectra[
#                 healthy_lookup[metabolite]
#             ]
#         )

#         axis.plot(
#             ppm_axis,
#             tumor_spectrum,
#             linestyle="-",
#             label="Tumor basis",
#         )

#         axis.plot(
#             ppm_axis,
#             healthy_spectrum,
#             linestyle="--",
#             label="Healthy basis",
#         )

#         axis.axhline(
#             0.0,
#             linewidth=0.5,
#             alpha=0.5,
#         )

#         axis.set_title(
#             metabolite
#         )

#         axis.set_xlabel(
#             "ppm"
#         )

#         axis.grid(
#             alpha=0.2
#         )

#         if (
#             ppm_min is not None
#             and ppm_max is not None
#         ):
#             axis.set_xlim(
#                 ppm_max,
#                 ppm_min,
#             )
#         else:
#             axis.invert_xaxis()

#         tumor_max = np.max(
#             np.abs(
#                 tumor_spectra[
#                     tumor_lookup[metabolite]
#                 ]
#             )
#         )

#         healthy_max = np.max(
#             np.abs(
#                 healthy_spectra[
#                     healthy_lookup[metabolite]
#                 ]
#             )
#         )

#         amplitude_ratios[metabolite] = (
#             tumor_max / healthy_max
#             if healthy_max > 0
#             else np.nan
#         )

#     for axis in flat_axes[n_metabolites:]:
#         axis.set_visible(False)

#     flat_axes[0].legend()

#     figure.suptitle(
#         f"Common metabolites: tumor vs healthy basis "
#         f"({component})",
#         y=1.0,
#     )

#     figure.tight_layout()

#     print(
#         f"Common metabolites ({len(common_metabolites)}):"
#     )
#     print(
#         ", ".join(common_metabolites)
#     )

#     print(
#         f"\nOnly in tumor basis ({len(tumor_only)}):"
#     )
#     print(
#         ", ".join(tumor_only)
#         if tumor_only
#         else "None"
#     )

#     print(
#         f"\nOnly in healthy basis ({len(healthy_only)}):"
#     )
#     print(
#         ", ".join(healthy_only)
#         if healthy_only
#         else "None"
#     )

#     print(
#         "\nPeak-magnitude ratio "
#         "(tumor / healthy):"
#     )

#     for metabolite, ratio in amplitude_ratios.items():
#         print(
#             f"  {metabolite:10s}: {ratio:.6f}"
#         )

#     return (
#         figure,
#         axes,
#         amplitude_ratios,
#     )

In [ ]:
# fig, axes, amplitude_ratios = compare_prepared_bases(
#     basis_tumor=prepared_basis_tumor,
#     basis_healthy=prepared_basis_healthy,
#     component="real",
#     n_columns=4,
#     ppm_min=0.0,
#     ppm_max=7.0,
# )